In [1]:
import sys
from scipy.stats import norm
import numpy as np
sys.path.append('..')  # so we can import from models/
from models.mc_greeks import bsm_greeks_pw

In [2]:
def analytic_delta_vega(S, K, T, r, sigma):
    """Closed-form call Delta and Vega — the validation truth."""
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    delta = norm.cdf(d1)
    vega = S * norm.pdf(d1) * np.sqrt(T)
    return delta, vega

In [3]:
# --- canonical ATM test case ---
S, K, T, r, sigma = 100.0, 100.0, 1.0, 0.05, 0.20
n_paths, seed = 1_000_000, 42

pw = bsm_greeks_pw(S, K, T, r, sigma, n_paths=n_paths, seed=seed)
delta_true, vega_true = analytic_delta_vega(S, K, T, r, sigma)

for name, truth in [("delta", delta_true), ("vega", vega_true)]:
    est, se = pw[name]
    z = abs(est - truth) / se
    flag = "ok" if z < 3 else "** CHECK **"
    print(f"{name:6s}  PW = {est:10.5f} ± {se:.5f}   "
          f"analytic = {truth:10.5f}   z = {z:5.2f}   {flag}")

delta   PW =    0.63692 ± 0.00020   analytic =    0.63683   z =  0.43   ok
vega    PW =   37.56368 ± 0.06616   analytic =   37.52403   z =  0.60   ok


Delta's is 0.0002 / 0.637 ≈ 0.03%. Vega's is 0.066 / 37.5 ≈ 0.18%, six times noisier for the same path count. That's the antithetic asymmetry showing up empirically: Delta's integrand is near-symmetric in Z, so the antithetic pairing cancels hard, while Vega's $(\sqrt(T)Z - \sigma T)$ weight flips sign with
Z and cancels far less.